# 📊 Notebook 01: WRI Aqueduct 4.0 Data Extraction

**Objective:** Download, explore, and extract water stress indicators from the WRI Aqueduct 4.0 dataset.

**Source:** [WRI Aqueduct Water Risk Atlas](https://www.wri.org/aqueduct)

**Key Indicators:**
- Baseline Water Stress (BWS)
- Water Depletion
- Interannual & Seasonal Variability
- Groundwater Table Decline
- Drought Risk

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

import config
from src.extractors.aqueduct_extractor import AqueductExtractor

# Set style
plt.style.use('dark_background')
sns.set_theme(style='darkgrid', palette='viridis')
pd.set_option('display.max_columns', 50)

print('✅ Setup complete')

## 1. Data Download
Download the Aqueduct 4.0 baseline dataset. If the WRI server is unavailable,
realistic synthetic data will be generated automatically.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

extractor = AqueductExtractor()

# Download baseline annual data
extractor.download_data('baseline_annual')

# Download country rankings
extractor.download_data('country_rankings')

print('\n✅ Data download complete')

## 2. Raw Data Exploration

In [ ]:
# Load raw data
raw_df = extractor.load_raw_data('baseline_annual')

print(f'Shape: {raw_df.shape}')
print(f'\nColumns ({len(raw_df.columns)}):\n{list(raw_df.columns)}')
print(f'\nData types:\n{raw_df.dtypes.value_counts()}')
print(f'\nMissing values:\n{raw_df.isnull().sum()[raw_df.isnull().sum() > 0]}')

raw_df.head(10)

In [ ]:
# Basic statistics
score_cols = [c for c in raw_df.columns if '_score' in c or '_raw' in c]
raw_df[score_cols].describe().round(3)

## 3. Feature Extraction

In [ ]:
# Extract standardized features
df = extractor.extract_features(raw_df)

print(f'Processed shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
df.head(10)

## 4. Visualization

In [ ]:
# Global Water Stress Map
if 'country_code' in df.columns and 'water_stress_score' in df.columns:
    country_avg = df.groupby(['country_code', 'country'])['water_stress_score'].mean().reset_index()
    
    fig = px.choropleth(
        country_avg,
        locations='country_code',
        locationmode='ISO-3',
        color='water_stress_score',
        hover_name='country',
        color_continuous_scale='RdYlBu_r',
        title='Global Baseline Water Stress (WRI Aqueduct 4.0)',
        labels={'water_stress_score': 'Water Stress Score (0-5)'},
    )
    fig.update_layout(
        geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
        height=500, template='plotly_dark',
    )
    fig.show()

In [ ]:
# Distribution of Water Stress Scores
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

score_columns = ['water_stress_score', 'drought_risk_score', 
                 'groundwater_decline_score', 'water_depletion_score']
colors = ['#3498db', '#e74c3c', '#f39c12', '#2ecc71']

for i, (col, color) in enumerate(zip(score_columns, colors)):
    ax = axes[i // 2, i % 2]
    if col in df.columns:
        ax.hist(df[col].dropna(), bins=40, color=color, alpha=0.7, edgecolor='white', linewidth=0.5)
        ax.set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
        ax.set_xlabel('Score (0-5)')
        ax.set_ylabel('Frequency')
        ax.axvline(df[col].mean(), color='white', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.2f}')
        ax.legend()

plt.suptitle('Distribution of Water Risk Indicators', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 most water-stressed countries
if 'country' in df.columns and 'water_stress_score' in df.columns:
    top20 = df.groupby('country')['water_stress_score'].mean().sort_values(ascending=False).head(20)
    
    fig = px.bar(
        x=top20.values, y=top20.index,
        orientation='h',
        title='Top 20 Most Water-Stressed Countries',
        labels={'x': 'Average Water Stress Score', 'y': 'Country'},
        color=top20.values,
        color_continuous_scale='RdYlBu_r',
    )
    fig.update_layout(template='plotly_dark', height=600, yaxis=dict(autorange='reversed'))
    fig.show()

In [ ]:
# Correlation matrix
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Save Processed Data

In [ ]:
# Save to processed directory
output_path = extractor.save_processed(df)
print(f'\n✅ Processed data saved to: {output_path}')
print(f'   Shape: {df.shape}')
print(f'   Countries: {df["country"].nunique() if "country" in df.columns else "N/A"}')

# Summary stats by country
summary = extractor.get_summary_stats(df)
print(f'\nCountry-level summary (top 10 by water stress):')
print(summary.sort_values(('water_stress_score', 'mean'), ascending=False).head(10))